# Retrain on the 3-class dataset (Elliptical / Merger / Spiral) — Weeks 4, 5, 6 redo
**TMI-102 | PaAC, IIT Roorkee** — runs on Kaggle (GPU on)

### What changed compared with the 2-class runs
- **Data:** `Galaxy_dataset_3class` — elliptical and spiral are **exactly the Week 3 dataset** (8,132 each; 5,692 / 1,220 / 1,220), and **merger is simply added** as a third class (Galaxy Zoo `Class8.6 ≥ 0.30`, skipping galaxies already used as elliptical/spiral so no image has two labels), split 70/15/15.
- **Output layer:** `Dense(3)`.
- **Imbalance:** mergers are ~3× rarer. The loss uses **class weights** (a rare-class mistake costs more) — same idea as raising the chance of drawing the rare class, but done in the loss.
- **Preprocessing:** `preprocess_input` everywhere (no `1./255`), same as your fixed Week 6.
- **Fair comparison:** all three models use the same data, augmentation (rotation 360°, flips), batch size and class weights. Only the thing being studied changes (frozen vs full vs LoRA, and the learning rate that goes with each).
- **All three models are saved** (`w4_frozen_3class`, `w5_full_3class`, `w6_lora_3class`) so Week 9 can compare them.

### Kaggle setup (nothing to upload from your laptop)
1. Add the **Galaxy Zoo competition data** to this notebook (same as your `week5 model` notebook), turn the GPU on.
2. Run top to bottom: the first code cell builds the 3-class dataset straight from the competition zips, then each model has its own cell.
3. After the run, "Save Version" so the output (3 `.keras` models + `Galaxy_dataset_3class`) can be added as an input to the Week 8 and Week 9 notebooks.

In [ ]:
# ── Build Galaxy_dataset_3class from the competition files (runs once; skipped if the folder already exists) ──
import os, zipfile
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge"
BUILD_DIR = "/kaggle/working/Galaxy_dataset_3class"
ELL_T, SPI_T, MERGER_T, SEED = 0.80, 0.80, 0.30, 42     # merger = Class8.6 >= 0.30 (change here if mentors say otherwise)

if not os.path.isdir(BUILD_DIR + "/train"):
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(f"{COMP}/training_solutions_rev1.zip", compression="zip")

    # --- elliptical + spiral: EXACTLY the Week 3 recipe (same thresholds, same seed, same split) ---
    df["class"] = np.select([df["Class1.1"] >= ELL_T, df["Class1.2"] >= SPI_T], ["elliptical", "spiral"], default="Tie")
    filtered = df[df["class"].isin(["elliptical", "spiral"])].copy()
    min_cnt = min(filtered["class"].value_counts())
    balanced = pd.concat([
        filtered[filtered["class"] == "spiral"].sample(n=min_cnt, random_state=SEED),
        filtered[filtered["class"] == "elliptical"].sample(n=min_cnt, random_state=SEED)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    tr, tmp = train_test_split(balanced, test_size=0.30, random_state=SEED, stratify=balanced["class"])
    va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp["class"])

    # --- merger: simply ADDED as a third class. Galaxies already used as elliptical/spiral are skipped,
    #     so one image never has two labels and the two old classes stay exactly as they were. ---
    mer_all = df[df["Class8.6"] >= MERGER_T]
    mer = mer_all[~mer_all["GalaxyID"].isin(balanced["GalaxyID"])].copy()
    mer["class"] = "merger"
    print(f"merger candidates {len(mer_all)} -> {len(mer)} after skipping {len(mer_all) - len(mer)} already used as elliptical/spiral")
    mtr, mtmp = train_test_split(mer, test_size=0.30, random_state=SEED)
    mva, mte = train_test_split(mtmp, test_size=0.50, random_state=SEED)

    splits = [(tr, "train"), (va, "val"), (te, "test"), (mtr, "train"), (mva, "val"), (mte, "test")]
    for s in ["train", "val", "test"]:
        for c in ["elliptical", "merger", "spiral"]:
            os.makedirs(f"{BUILD_DIR}/{s}/{c}", exist_ok=True)
    target, rows = {}, []
    for d, s in splits:
        for gid, c in zip(d["GalaxyID"], d["class"]):
            target[f"{gid}.jpg"] = (s, c); rows.append((gid, c, s))

    n_ext = 0
    with zipfile.ZipFile(f"{COMP}/images_training_rev1.zip") as z:
        for member in z.namelist():
            base = os.path.basename(member)
            if base in target:
                s, c = target[base]
                with open(f"{BUILD_DIR}/{s}/{c}/{base}", "wb") as f:
                    f.write(z.read(member))
                n_ext += 1
    man = pd.DataFrame(rows, columns=["GalaxyID", "cls", "split"])
    man.to_csv(f"{BUILD_DIR}/manifest.csv", index=False)
    print("extracted", n_ext, "of", len(man))
    print(man.groupby(["split", "cls"]).size().unstack())
    print("Check vs Week 3: elliptical/spiral should be 5692 / 1220 / 1220 each.")
else:
    print("Dataset already built at", BUILD_DIR)

In [ ]:
import os, glob, json, time
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("TF", tf.__version__, "| GPU:", len(tf.config.list_physical_devices('GPU')) > 0)

DATASET_DIR = BUILD_DIR
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print("Dataset:", DATASET_DIR)

In [ ]:
IMG_SIZE, BATCH_SIZE, NUM_CLASSES = 224, 32, 3

train_aug = ImageDataGenerator(preprocessing_function=preprocess_input,
                               rotation_range=360, horizontal_flip=True, vertical_flip=True)
plain = ImageDataGenerator(preprocessing_function=preprocess_input)

def make(gen, split, shuffle):
    return gen.flow_from_directory(f'{DATASET_DIR}/{split}', target_size=(IMG_SIZE, IMG_SIZE),
                                   batch_size=BATCH_SIZE, class_mode='categorical', shuffle=shuffle)

train_gen = make(train_aug, 'train', True)
val_gen   = make(plain, 'val', False)
test_gen  = make(plain, 'test', False)

print(train_gen.class_indices)          # expect {'elliptical': 0, 'merger': 1, 'spiral': 2}
CLASS_NAMES = [c for c, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]

# class weight = N / (K * count_of_that_class)  -> rare class counts more in the loss
counts = np.bincount(train_gen.classes)
class_weight = {i: len(train_gen.classes) / (NUM_CLASSES * n) for i, n in enumerate(counts)}
for i, c in enumerate(CLASS_NAMES):
    print(f"{c:>10}: {counts[i]:5d} train images -> weight {class_weight[i]:.3f}")

In [ ]:
@tf.keras.utils.register_keras_serializable(package='tmi102')
class LoRADense(layers.Layer):
    """Dense layer with a LoRA adapter: output = x@W + b + (x@A@B) * (alpha/rank).  W, b frozen; A, B trainable."""
    def __init__(self, units, rank=4, alpha=32, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units, self.rank, self.alpha = units, rank, alpha
        self.scaling = alpha / rank
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(name='kernel', shape=(input_dim, self.units), initializer='glorot_uniform', trainable=False)
        self.b = self.add_weight(name='bias', shape=(self.units,), initializer='zeros', trainable=False)
        self.A = self.add_weight(name='lora_A', shape=(input_dim, self.rank), initializer='glorot_normal', trainable=True)
        self.B = self.add_weight(name='lora_B', shape=(self.rank, self.units), initializer='zeros', trainable=True)

    def call(self, x):
        h = tf.matmul(x, self.W) + self.b
        h = h + tf.matmul(tf.matmul(x, self.A), self.B) * self.scaling
        return self.activation_fn(h) if self.activation_fn else h

    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units, rank=self.rank, alpha=self.alpha,
                   activation=tf.keras.activations.serialize(self.activation_fn))
        return cfg

In [ ]:
def build_model(mode):
    """mode: 'frozen' (Week 4) | 'full' (Week 5) | 'lora' (Week 6)"""
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = (mode == 'full')
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs) if mode == 'full' else base(inputs, training=False)   # frozen/LoRA: BatchNorm stays in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    if mode == 'lora':
        x = LoRADense(256, rank=4, alpha=32, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
    else:
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inputs, outputs, name=f'resnet50_{mode}')

RESULTS_PATH = f'{OUT_DIR}/results_3class.json'

def run(mode, tag, lr, epochs, patience, reduce_lr):
    tf.keras.backend.clear_session()
    model = build_model(mode)
    n_train = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    n_total = int(model.count_params())
    print(f"[{tag}] trainable {n_train:,} / total {n_total:,} ({n_train / n_total * 100:.2f}%)")

    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    cbs = [EarlyStopping(patience=patience, restore_best_weights=True)]
    if reduce_lr:
        cbs.append(ReduceLROnPlateau(patience=3, factor=0.5, verbose=1))

    t0 = time.time()
    hist = model.fit(train_gen, validation_data=val_gen, epochs=epochs, class_weight=class_weight, callbacks=cbs)
    elapsed = time.time() - t0
    n_ep = len(hist.history['loss'])

    test_loss, test_acc = model.evaluate(test_gen, verbose=0)
    test_gen.reset()
    probs = model.predict(test_gen, verbose=0)
    y, p = test_gen.classes, probs.argmax(1)
    cm = confusion_matrix(y, p)
    macro_f1 = f1_score(y, p, average='macro')
    recall = cm.diagonal() / cm.sum(1)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(hist.history['accuracy'], label='Train'); a1.plot(hist.history['val_accuracy'], label='Val')
    a1.set_title(f'{tag} — Accuracy'); a1.legend(); a1.grid(True)
    a2.plot(hist.history['loss'], label='Train'); a2.plot(hist.history['val_loss'], label='Val')
    a2.set_title(f'{tag} — Loss'); a2.legend(); a2.grid(True)
    plt.tight_layout(); plt.show()

    print(f"\nTest accuracy {test_acc * 100:.2f}% | macro-F1 {macro_f1:.4f} | {n_ep} epochs | {elapsed / n_ep:.0f}s/epoch")
    print("Confusion matrix (rows = true, cols = predicted):", CLASS_NAMES)
    print(cm)
    print(classification_report(y, p, target_names=CLASS_NAMES, digits=3))

    model.save(f'{OUT_DIR}/{tag}.keras')
    np.save(f'{OUT_DIR}/{tag}_test_probs.npy', probs)
    res = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
    res[tag] = dict(trainable_params=n_train, total_params=n_total, test_acc=float(test_acc), macro_f1=float(macro_f1),
                    recall={c: float(r) for c, r in zip(CLASS_NAMES, recall)}, epochs_run=n_ep,
                    sec_per_epoch=float(elapsed / n_ep), lr=lr)
    json.dump(res, open(RESULTS_PATH, 'w'), indent=2)
    return model

## Week 4 redo — frozen ResNet-50 (feature extraction)
Only the new head trains. LR 1e-3 (the head starts from scratch, so it can move fast).

In [ ]:
model4 = run('frozen', 'w4_frozen_3class', lr=1e-3, epochs=15, patience=5, reduce_lr=True)

## Week 5 redo — full fine-tuning
Everything unfrozen, LR 1e-4 (small steps so ImageNet features are adapted, not destroyed).

In [ ]:
model5 = run('full', 'w5_full_3class', lr=1e-4, epochs=15, patience=5, reduce_lr=True)

## Week 6 redo — LoRA head (rank 4)
Backbone frozen; the 256-unit head layer is a `LoRADense` (only A and B train). LR 1e-3.

In [ ]:
model6 = run('lora', 'w6_lora_3class', lr=1e-3, epochs=12, patience=4, reduce_lr=False)

## Final comparison table (numbers come from `results_3class.json`)

In [ ]:
res = json.load(open(RESULTS_PATH))
names = {'w4_frozen_3class': 'Week 4: Frozen', 'w5_full_3class': 'Week 5: Full FT', 'w6_lora_3class': 'Week 6: LoRA (r=4)'}
print(f"{'Model':<20}{'Params trained':>16}{'Test acc':>10}{'Macro-F1':>10}{'s/epoch':>9}   recall per class {CLASS_NAMES}")
for k, label in names.items():
    r = res[k]
    rec = "  ".join(f"{r['recall'][c] * 100:5.1f}%" for c in CLASS_NAMES)
    print(f"{label:<20}{r['trainable_params']:>16,}{r['test_acc'] * 100:>9.2f}%{r['macro_f1']:>10.4f}{r['sec_per_epoch']:>9.0f}   {rec}")
print("\nFiles in", OUT_DIR, ":", sorted(f for f in os.listdir(OUT_DIR) if f.endswith(('.keras', '.json', '.npy'))))